# RavenStack SaaS Analytics

This notebook performs the core data scientist workflow for RavenStack: load raw CSV data, clean and merge tables, engineer SaaS metrics, and verify results with summary tables and charts.


In [ ]:
# Install required libraries if they are not already installed
# Uncomment the following line when running this notebook in a fresh environment:
# !pip install pandas numpy plotly streamlit

import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path


In [ ]:
data_dir = Path('Data_file')
accounts = pd.read_csv(data_dir / 'ravenstack_accounts.csv', parse_dates=['signup_date'])
subscriptions = pd.read_csv(data_dir / 'ravenstack_subscriptions.csv', parse_dates=['start_date', 'end_date'])
usage = pd.read_csv(data_dir / 'ravenstack_feature_usage.csv', parse_dates=['usage_date'])
tickets = pd.read_csv(data_dir / 'ravenstack_support_tickets.csv', parse_dates=['submitted_at', 'closed_at'])
churn = pd.read_csv(data_dir / 'ravenstack_churn_events.csv', parse_dates=['churn_date'])

print('Accounts shape:', accounts.shape)
print('Subscriptions shape:', subscriptions.shape)
print('Usage shape:', usage.shape)
print('Support tickets shape:', tickets.shape)
print('Churn events shape:', churn.shape)

accounts.head()


In [ ]:
def bool_cast(series):
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])

def clean_accounts(accounts):
    df = accounts.copy()
    df['industry'] = df['industry'].astype(str).str.strip().str.title()
    df['country'] = df['country'].astype(str).str.strip().str.upper()
    df['plan_tier'] = df['plan_tier'].astype(str).str.strip().str.title()
    df['referral_source'] = df['referral_source'].astype(str).str.strip().str.lower()
    df['seats'] = pd.to_numeric(df['seats'], errors='coerce').fillna(0).astype(int)
    df['is_trial'] = bool_cast(df['is_trial'])
    df['churn_flag'] = bool_cast(df['churn_flag'])
    df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
    return df

def clean_subscriptions(subscriptions):
    df = subscriptions.copy()
    df['plan_tier'] = df['plan_tier'].astype(str).str.strip().str.title()
    df['billing_frequency'] = df['billing_frequency'].astype(str).str.strip().str.lower()
    df['mrr_amount'] = pd.to_numeric(df['mrr_amount'], errors='coerce').fillna(0)
    df['arr_amount'] = pd.to_numeric(df['arr_amount'], errors='coerce').fillna(0)
    for col in ['is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'auto_renew_flag']:
        df[col] = bool_cast(df[col])
    df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
    df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')
    return df

def clean_usage(usage):
    df = usage.copy()
    df['feature_name'] = df['feature_name'].astype(str).str.strip().str.lower()
    df['usage_count'] = pd.to_numeric(df['usage_count'], errors='coerce').fillna(0).astype(int)
    df['usage_duration_secs'] = pd.to_numeric(df['usage_duration_secs'], errors='coerce').fillna(0)
    df['error_count'] = pd.to_numeric(df['error_count'], errors='coerce').fillna(0).astype(int)
    df['is_beta_feature'] = bool_cast(df['is_beta_feature'])
    df['usage_date'] = pd.to_datetime(df['usage_date'], errors='coerce')
    df['session_length_secs'] = np.where(df['usage_count'] > 0, df['usage_duration_secs'] / df['usage_count'], 0)
    df['friction_score'] = np.where(df['usage_count'] > 0, df['error_count'] / df['usage_count'], 0)
    return df

def clean_tickets(tickets):
    df = tickets.copy()
    df['priority'] = df['priority'].astype(str).str.strip().str.title()
    df['satisfaction_score'] = pd.to_numeric(df['satisfaction_score'], errors='coerce')
    df['escalation_flag'] = bool_cast(df['escalation_flag'])
    df['submitted_at'] = pd.to_datetime(df['submitted_at'], errors='coerce')
    df['closed_at'] = pd.to_datetime(df['closed_at'], errors='coerce')
    df['ticket_latency_hours'] = (df['closed_at'] - df['submitted_at']).dt.total_seconds() / 3600
    return df

def clean_churn(churn):
    df = churn.copy()
    df['reason_code'] = df['reason_code'].astype(str).str.strip().str.lower()
    df['refund_amount_usd'] = pd.to_numeric(df['refund_amount_usd'], errors='coerce').fillna(0)
    df['preceding_upgrade_flag'] = bool_cast(df['preceding_upgrade_flag'])
    df['preceding_downgrade_flag'] = bool_cast(df['preceding_downgrade_flag'])
    df['is_reactivation'] = bool_cast(df['is_reactivation'])
    df['feedback_text'] = df['feedback_text'].fillna('')
    df['churn_date'] = pd.to_datetime(df['churn_date'], errors='coerce')
    return df


In [ ]:
accounts_clean = clean_accounts(accounts)
subscriptions_clean = clean_subscriptions(subscriptions)
usage_clean = clean_usage(usage)
tickets_clean = clean_tickets(tickets)
churn_clean = clean_churn(churn)

df_revenue = subscriptions_clean.merge(accounts_clean[['account_id', 'industry', 'country']], on='account_id', how='left')
df_usage_value = usage_clean.merge(subscriptions_clean[['subscription_id', 'mrr_amount', 'plan_tier', 'account_id']], on='subscription_id', how='left')
df_tickets = tickets_clean.merge(accounts_clean[['account_id', 'industry', 'country']], on='account_id', how='left')
df_churn = churn_clean.merge(accounts_clean[['account_id', 'industry', 'country']], on='account_id', how='left')

total_mrr = subscriptions_clean['mrr_amount'].sum()
active_subscriptions = subscriptions_clean.loc[subscriptions_clean['churn_flag'] == False].shape[0]
churn_rate = (df_churn.shape[0] / max(accounts_clean.shape[0], 1)) * 100
avg_sat = tickets_clean['satisfaction_score'].mean()

print('Total MRR:', total_mrr)
print('Active Subscriptions:', active_subscriptions)
print('Churn Rate (%):', round(churn_rate, 2))
print('Average Satisfaction Score:', round(avg_sat, 2))


In [ ]:
industry_revenue = df_revenue.groupby('industry')['mrr_amount'].sum().reset_index().sort_values('mrr_amount', ascending=False)
fig = px.bar(
    industry_revenue,
    x='industry',
    y='mrr_amount',
    title='Revenue by Industry',
    text='mrr_amount',
)
fig.update_traces(texttemplate='$%{text:.0f}', textposition='outside')
fig.show()

churn_by_reason = df_churn['reason_code'].value_counts().rename_axis('reason_code').reset_index(name='count')
fig2 = px.pie(churn_by_reason, names='reason_code', values='count', title='Churn Reason Distribution', hole=0.4)
fig2.show()

feature_stats = df_usage_value.groupby('feature_name')[['usage_count', 'error_count']].sum().reset_index()
fig3 = px.scatter(
    feature_stats,
    x='usage_count',
    y='error_count',
    size='error_count',
    color='error_count',
    text='feature_name',
    title='Feature usage versus error count',
)
fig3.update_traces(textposition='top center')
fig3.show()
